# P28 — El prompting de cadena de pensamiento provoca razonamiento en modelos de lenguaje grandes

## 1. Título y paper

**Paper:** *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*  
**Autoría:** Jason Wei, Xuezhi Wang, Dale Schuurmans, Maarten Bosma, Brian Ichter, Fei Xia, Ed Chi, Quoc Le, Denny Zhou  
**Año y venue:** 2022 · arXiv:2201.11903 · NeurIPS 2022  
**Nivel:** L2 · **Motor:** `cot`  
**Ficha completa:** [`P28_chain_of_thought`](../../papers/foundational/P28_chain_of_thought/README.md)

**Hito:** Descomponer en pasos intermedios desbloquea tareas que el mismo modelo fallaba respondiendo de una vez.

- [arXiv:2201.11903](https://arxiv.org/abs/2201.11903)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los modelos grandes fallaban en aritmética y razonamiento de varios pasos aunque acertaran tareas aparentemente más difíciles: se les pedía el resultado sin dejarles espacio para llegar a él.
2. Ejecutar una implementación mínima de la propuesta: Incluir en el prompt unos pocos ejemplos que muestren el razonamiento paso a paso, sin ajuste fino ni datos adicionales.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10


## 4. Intuición

Pedir el resultado de una multiplicación de tres cifras «de cabeza» falla; pedirla por pasos, no. No es que el modelo sepa más: es que le has dado sitio donde hacer la cuenta.


## 5. Concepto mínimo

```text
Directo:   pregunta → respuesta
Cadena :   pregunta → paso 1 → paso 2 → … → respuesta
```

La aritmética de por qué funciona:

```text
P(acertar directo)  ≈ dificultad del problema entero
P(acertar cadena)   ≈ (fiabilidad por paso)^n
```

Descomponer gana **si y solo si** cada paso es mucho más fiable que el problema completo. Por eso no funciona en modelos pequeños: sus pasos no son suficientemente buenos.


## 6. Código explicado

El motor modela ambas probabilidades y localiza el umbral de fiabilidad por paso.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('cot', seed=7)['result']
show(r['supuestos'])
print()
for f in r['por_numero_de_pasos']:
    print(f"{f['pasos']:>2} pasos · directo {f['directo']:.4f} · cadena {f['cadena']:.4f} "
          f"· gana cadena: {f['gana_cadena']}")

## 7. Predicción antes de ejecutar

1. ¿La cadena gana siempre, o hay un número de pasos a partir del cual pierde?
2. ¿De qué depende realmente que compense: del número de pasos o de la calidad de cada uno?
3. ¿Por qué el efecto no aparece en modelos pequeños?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('cot', seed=7)['result']
print('umbral de fiabilidad por paso:', r['fiabilidad_por_paso_minima_para_que_compense'])
print()
for e in r['emergencia_con_la_escala']:
    print(f"{e['parametros_miles_millones']:>6} MM parámetros · cadena {e['cadena_3_pasos']:.4f} "
          f"· directo {e['directo']:.4f} · ayuda: {e['la_cadena_ayuda']}")

## 9. Salida interpretable

El cruce **no está en el número de pasos** sino en la fiabilidad de cada uno. Por debajo del umbral, descomponer empeora: multiplicas errores. Por encima, mejora. Y como la fiabilidad por paso crece con la escala del modelo, el efecto **emerge**: no es que aparezca una capacidad mágica, es que un producto de números cruza un umbral.


## 10. Comentario pedagógico

Ojo con la palabra «emergencia». Aquí se puede explicar con probabilidad elemental. Trabajo posterior discutió si muchas capacidades «emergentes» lo son de verdad o son artefactos de métricas discontinuas (acertar/fallar) que ocultan una mejora continua.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer la cadena como una explicación del proceso interno del modelo.


In [ ]:
print('La cadena es TEXTO GENERADO, optimizado para que la respuesta final sea correcta.')
print('Puede contener pasos invalidos y llegar al resultado correcto, o al reves.')
print('Sirve para depurar y para dar sitio al calculo; no es un certificado.')

## 12. Corrección

Lo que sí se puede afirmar, y cómo comprobarlo:


In [ ]:
auditoria = {
    'comprobable': 'la respuesta final, contra la solución conocida',
    'no_comprobable_sin_trabajo': 'la validez de cada paso intermedio',
    'como_reforzarlo': ['muestrear varias cadenas y votar (autoconsistencia)',
                         'ejecutar el cálculo con una herramienta externa'],
}
show(auditoria)

## 13. Desafío guiado

Calcula cuántos pasos aguanta una cadena antes de bajar del 50 % de fiabilidad, para varias calidades por paso.


In [ ]:
import math
for q in (0.80, 0.90, 0.95, 0.99):
    n = math.floor(math.log(0.5) / math.log(q))
    print(f'fiabilidad por paso {q:.2f} → aguanta {n:>3} pasos por encima del 50%')

## 14. Desafío autónomo

Con un modelo abierto pequeño y un conjunto de problemas aritméticos, compara respuesta directa frente a cadena de pensamiento. Mide además cuántas cadenas contienen un paso inválido pero llegan al resultado correcto: esa tasa es la que desmonta la lectura ingenua.


## 15. Evidencia de aprendizaje

Guarda la tabla directo/cadena, el umbral de fiabilidad y tu explicación de por qué la emergencia aquí es aritmética y no magia.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P28_chain_of_thought/README.md) · evaluación formal: [`assessments/papers/P28_chain_of_thought.md`](../../assessments/papers/P28_chain_of_thought.md)


## 16. Cierre

Razonar en línea recta ayuda, pero no permite volver atrás. Si un paso intermedio es malo, toda la cadena se pierde — y ahí entra la búsqueda.


## 17. Conexión con el siguiente hito

- P13
- P29
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
